In [1]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_skewed_table
import time

spark = get_spark("Case06_DataSkewness", {
    "spark.sql.adaptive.enabled": "false"  
})


26/04/14 11:04:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# STEP 1: Detect Skew

In [2]:
df = generate_skewed_table(spark, num_rows=2_000_000, hot_keys=3, hot_key_ratio=0.7)

In [3]:
print("Method 1: Top keys by row count")
key_stats=(
    df.groupBy("account_number")
    .count()
    .orderBy(F.desc("count"))
)
key_stats.show(10)

Method 1: Top keys by row count


+--------------+------+
|account_number| count|
+--------------+------+
|             0|466667|
|             1|466667|
|             2|466666|
|           119|     6|
|            45|     6|
|           898|     6|
|           254|     6|
|          1001|     6|
|            57|     6|
|          1389|     6|
+--------------+------+
only showing top 10 rows



In [4]:
print("Method 2: Percentile analysis")
key_counts=df.groupBy("account_number").count()
key_counts.select(
    F.expr("percentile_approx(count,0.5)").alias("p50_median"),
    F.expr("percentile_approx(count, 0.75)").alias("p75"),
    F.expr("percentile_approx(count, 0.95)").alias("p95"),
    F.expr("percentile_approx(count, 0.99)").alias("p99"),
    F.max("count").alias("max"),
    F.min("count").alias("min")
).show()

Method 2: Percentile analysis


+----------+---+---+---+------+---+
|p50_median|p75|p95|p99|   max|min|
+----------+---+---+---+------+---+
|         6|  6|  6|  6|466667|  6|
+----------+---+---+---+------+---+



In [5]:
print("Method 3: Partition size distribution")
partition_sizes=df.rdd.mapPartitions(lambda it:[sum(1 for _ in it)]).collect()
print(f"  Min partition:  {min(partition_sizes):,} rows")
print(f"  Max partition:  {max(partition_sizes):,} rows")
print(f"  Skew ratio: {max(partition_sizes)/max(min(partition_sizes),1):.1f}x")


Method 3: Partition size distribution


  Min partition:  37,500 rows
  Max partition:  87,500 rows
  Skew ratio: 2.3x


# STEP 2: Fix with AQE Skew Join

In [6]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64MB")

df2=df.groupBy("account_number").agg(F.avg("value").alias("avg_value"))

In [7]:
start=time.time()
result=df.join(df2,"account_number","inner")
result.write.mode("overwrite").parquet("/tmp/case06_aqe_skew")
print(f"\nAQE Skew Join: {time.time() - start:.1f}s")

26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/14 11:05:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers



AQE Skew Join: 3.4s


26/04/14 11:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/14 11:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/14 11:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/14 11:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/14 11:05:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


# STEP 3: Fix with manual salting for joins

In [10]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [10]:
SALT_BUCKETS = 10
salted_large=df.withColumn("salt",(F.rand()*SALT_BUCKETS).cast("int"))
salted_small = (
    df2
    .crossJoin(spark.range(0, SALT_BUCKETS).withColumnRenamed("id", "salt"))
)

start=time.time()
result_salted=salted_large.join(
    salted_small,
    ["account_number","salt"],
    "inner"
).drop("salt")
result_salted.write.mode("overwrite").parquet("/tmp/case06_salted_join")
print(f"Salted Join: {time.time()-start:.1f}s")

26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/14 11:11:36 WARN MemoryManager: Total allocation exceeds 95.

Salted Join: 3.7s
